# Image Classification Workflow via API

**How to evaluate common workflows via the RI API**

This notebook demonstrates a full end to end workflow of the RI backend. The general steps are:

* Create wrapper objects as necessary (not all wrappers are needed for every tool)
  * Create a model wrapper
  * Create a dataset wrapper
  * Create a metric wrapper
* Create configuration to define analyses to run
* Generate "Test Stage" object(s)
* Run analyses

All RI wrappers are maite-compliant (https://jatic.pages.jatic.net/cdao/maite/). 

**IMPORTANT**:   
* **Since we are using synthetic, all black images, the values presented in the results will be meaningless.** Use full datasets with models trained on similar data to view accurate results. 
* This notebook requires a dev install since it utilizes data from the test suite.

In [ ]:
from jatic_ri import cache_path
from jatic_ri.util.utils import create_expandable_output

import os
import json
import torch
from pathlib import Path
from pprint import pprint as print
import warnings

for cat in (UserWarning, FutureWarning, RuntimeWarning):
    warnings.filterwarnings("ignore", category=cat)

### Set cache path 

In [ ]:
# the RI has a default output cache path, but it can be overriden as needed. The cache will only be written if `use_stage_cache=True` on the test stage.
print(f'The default cache path is {cache_path()}')

# set the cache path
cache_path('tscache')

print(f'The current cache path is {cache_path()}')

### Define task 

In [ ]:
task = 'image_classification'

### Create a model wrapper

Model wrappers hold the model object, weights and metadata.

The RI has one image classification model wrapper - `torchvision`  Below are examples of each of the available configuration options. 

#### Torchvision IC Model using default weights from Torchvision

In [ ]:
from jatic_ri.image_classification.models import TorchvisionICModel
model_name = "resnext50_32x4d"
model_wrapper = TorchvisionICModel(model_name=model_name)

#### Torchvision IC Model using custom weights and config

Note: The configuration file is a JSON formatted text file that contains `index2label` as a key with its value being a dictionary of {index: category label} for the model. 

In [ ]:
# save metadata and state_dict from previous cell to disk
config_path = cache_path() / "my_model.json"
pickle_path = cache_path() / "my_model.pt"

os.makedirs(os.path.dirname(config_path), exist_ok=True)
with open(config_path, "w") as f:
    json.dump({"index2label": model_wrapper.index2label}, f)
_ = torch.save(model_wrapper.model.state_dict(), pickle_path)

# extra kwargs to pass through to torchvision model object
kwargs = {}
model_wrapper = TorchvisionICModel(
    model_name=model_name,
    weights_path=pickle_path,
    config_path=config_path,
    model_id="arbitraryidnumber",
    **kwargs,
)

### Create dataset wrapper

Dataset wrappers control the access to the dataset and contain metadata about the dataset and about individual images.

The RI has one dataset wrapper for image classification - `YOLO`. In the cell below, a dummy YOLO-formatted dataset is generated, 
followed by an example of how to load it.

In [ ]:
from jatic_ri.image_classification.datasets import YoloClassificationDataset
from PIL import Image
classes = ["cat", "dog"]
num_images_per_class = 3
img_shape = (64, 128)

root_dir = Path('temp_yolo_dataset').resolve()
split = 'test'
os.makedirs(root_dir / split, exist_ok=True)
for class_name in classes:
    class_dir = root_dir / split / class_name
    os.makedirs(class_dir, exist_ok=True)
    for i in range(num_images_per_class):
        img = Image.new("RGB", img_shape, color=(i, i, i))
        img.save(class_dir / f"{i}_{class_name}.jpg")
         
dataset_wrapper = YoloClassificationDataset(
    dataset_id="temp_yolo_dataset", root_dir=root_dir, split="test"
)

### Create metric wrapper 

Metric wrappers provide standardized access to metric algorithms across the pydata ecosystem. 

The RI has two image classification metric wrappers - `Accuracy` and `F1 score`. 

In [ ]:
from jatic_ri.image_classification.metrics import accuracy_multiclass_torch_metric_factory
metric_wrapper = accuracy_multiclass_torch_metric_factory(num_classes=12)

### Instantiate and run each Test Stage

Each stage will be run and a list of slide data will be built

**NOTE**: Since we are using synthetic, all black images, the values presented in the results will be meaningless.

In [ ]:
# collect all the report slides in a list (to be convert to ppt after all stages are run)
slides = []

#### MAITE - Baseline evaluation 

In [ ]:
%%time
from jatic_ri.image_classification.test_stages import BaselineEvaluation

# instantiate test stage object from config values (no parameters necessary) 
test_stage = BaselineEvaluation()
# populate test_stage with data
test_stage.load_dataset(dataset_wrapper, dataset_id='dataset1')
test_stage.load_metric(metric_wrapper, metric_wrapper.return_key)
test_stage.load_threshold(0.5)
test_stage.load_model(model_wrapper, model_id='model1')

# run analysis for this test stage
base_eval_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
base_eval_run.outputs

#### NRTK 

In [ ]:
%%time
from jatic_ri.image_classification.test_stages import NRTKTestStage
from jatic_ri._common.test_stages import NRTKTestStageConfig

# define nrtk config
nrtk_config_dict = {
    'name': 'natural_robustness_TestFactory',
    'perturber_factory': {
        'type': 'nrtk.impls.perturb_image_factory.generic.one_step.OneStepPerturbImageFactory',
        'nrtk.impls.perturb_image_factory.generic.one_step.OneStepPerturbImageFactory': {
            'perturber': 'nrtk.impls.perturb_image.generic.PIL.enhance.BrightnessPerturber',
            'theta_key': 'factor',
            'theta_value': 10.0,
        },
    },
}
nrtk_config = NRTKTestStageConfig(**nrtk_config_dict)
# instantiate test stage object from config values
test_stage = NRTKTestStage(nrtk_config)
# populate test_stage with data
test_stage.load_dataset(dataset_wrapper, dataset_id='dataset1')
test_stage.load_metric(metric_wrapper, metric_wrapper.return_key)
test_stage.load_threshold(0.5)
test_stage.load_model(model_wrapper, model_id='model1')

# run analysis for this test stage
nrtk_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
nrtk_run.outputs

#### Survivor 

In [ ]:
%%time
from jatic_ri.image_classification.test_stages.impls.survivor_test_stage import SurvivorTestStage
from jatic_ri._common.test_stages.impls.survivor_test_stage import SurvivorConfig

# define survivor config 
survivor_config_dict = {
    'metric_column': 'metric',
    'conversion_type': 'original',
    'otb_threshold': 0.5,
    'easy_hard_threshold': 0.5,
}
survivor_config = SurvivorConfig(**survivor_config_dict)
# instantiate test stage object from config values
test_stage = SurvivorTestStage(survivor_config)
# populate test_stage with data
test_stage.load_dataset(dataset_wrapper, dataset_id='dataset1')
test_stage.load_metric(metric_wrapper, metric_wrapper.return_key)
test_stage.load_models({'model1': model_wrapper, 'model2': model_wrapper})

# run analysis for this test stage
survivor_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
create_expandable_output(survivor_run.outputs)

#### Dataeval - Bias

In [ ]:
%%time
from jatic_ri.image_classification.test_stages import DatasetBiasTestStage

# instantiate test stage object (no parameters necessary)
test_stage = DatasetBiasTestStage()
# populate test_stage with data
test_stage.load_dataset(dataset_wrapper, dataset_id='dataset1')

# run analysis for this test stage
bias_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

In [ ]:
# view results
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(bias_run.outputs.balance)

In [ ]:
create_expandable_output(bias_run.outputs.coverage)

In [ ]:
create_expandable_output(bias_run.outputs.diversity)

#### Dataeval - Feasibility

In [ ]:
%%time
from jatic_ri.image_classification.test_stages import DatasetImageClassificationFeasibilityTestStage

# instantiate test stage object (no parameters necessary)
test_stage = DatasetImageClassificationFeasibilityTestStage()
# populate test_stage with data
test_stage.load_dataset(dataset_wrapper, dataset_id='dataset1')
test_stage.load_threshold(0.5)

# run analysis for this test stage
feasibility_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
feasibility_run.outputs

#### Dataeval - Cleaning

In [ ]:
%%time
from jatic_ri.image_classification.test_stages import DatasetCleaningTestStage

# instantiate test stage object (no parameters necessary)
test_stage = DatasetCleaningTestStage()
# populate test_stage with data
test_stage.load_dataset(dataset_wrapper, dataset_id='dataset1')

# run analysis for this test stage
cleaning_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
cleaning_run.outputs
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(cleaning_run.outputs)

#### Dataeval - Shift

In [ ]:
%%time
from jatic_ri.image_classification.test_stages import DatasetShiftTestStage

# instantiate test stage object (no parameters necessary)
test_stage = DatasetShiftTestStage()
# populate test_stage with data
test_stage.load_datasets(dataset_1=dataset_wrapper, dataset_1_id='dataset1', dataset_2=dataset_wrapper, dataset_2_id='dataset2')

# run analysis for this test stage
shift_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# collect the slides for the final report
stage_slides = test_stage.collect_report_consumables()
# add to overall slide list
slides += stage_slides

# view results
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(shift_run.outputs)

#### XAITK

XAITK should only be run on single images or a datasets with only a few images due to memory constraints

In [ ]:
%%time
from jatic_ri.image_classification.test_stages.impls.xaitk_test_stage import XAITKTestStage, XAITKConfigIC

# define xaitk config
xaitk_config_dict = {
    'name': 'saliency_XAITKApp_0',
    'saliency_generator': {
        'type': 'xaitk_saliency.impls.gen_image_classifier_blackbox_sal.rise.RISEStack',
        'xaitk_saliency.impls.gen_image_classifier_blackbox_sal.rise.RISEStack': {
            'n': 50,
            's': 7,
            'p1': 0.7,
            'seed': 42,
            'threads': 8,
            'debiased': True,
        },
    },
    'img_batch_size': 1,
}
xaitk_config = XAITKConfigIC(**xaitk_config_dict)
# instantiate test stage object 
test_stage = XAITKTestStage(xaitk_config)
# populate test_stage with data
test_stage.load_dataset(dataset=dataset_wrapper, dataset_id='test')
test_stage.load_model(model_wrapper, model_id='model1')

# run analysis for this test stage
xaitk_run = test_stage.run(use_stage_cache=False)  # NOTE: `use_stage_cache=True` will bypass the compute and load cached results if available

# XAITK produces one slide per object per image. The example dataset is too large to generate a slidedeck of this size. 
# # collect the slides for the final report
# stage_slides = test_stage.collect_report_consumables()
# # add to overall slide list
# slides += stage_slides

# view results
# output is large, so expandable outputs are shown instead of raw outputs here
create_expandable_output(xaitk_run.outputs.results)

## Construct final report

In [ ]:
from gradient.templates_and_layouts.create_deck import create_deck
import copy 

# construct ppt report with summarized results
report_path = cache_path() / "report"
report_path.mkdir(parents=False, exist_ok=True)
report_filename = 'sample_report'
report = create_deck(copy.deepcopy(slides), report_path, deck_name=report_filename)

print(f'Report saved to {report}')

<hr>